# Project CV2: Robust Head Pose Estimation Under Facial Occlusions

Work by Antonio Vila Leis and Enric Baixauli Casañ

## Introduction

This project addresses the challenge of accurately estimating head posture in the presence of facial occlusions, such as masks and sunglasses.

TODO explicar todo

In [8]:
import os
import torch
import torchvision.transforms as T
from torch.utils.data import DataLoader
import sys
from dataset import LP300W
import utils
from tqdm import tqdm
from sklearn.model_selection import train_test_split
import torch.nn as nn
import torch.optim as optim
import time
from models.utils import compute_euler_angles_from_rotation_matrices

In [2]:
utils.set_global_seed(123)

if torch.cuda.is_available():
    device_name = "cuda"
elif torch.backends.mps.is_available():
    device_name = "mps"
else:
    device_name = "cpu"
    
device = torch.device(device_name)
print(f"Code runs in {device}")

Code runs in cuda


## 1. Data Loading

In [3]:
DATASET_ROOT = "./dataset/300W_LP"
BATCH_SIZE = 32

transforms_pipeline = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
])

all_image_paths = []
for folder in os.listdir(DATASET_ROOT):
    if folder.endswith('_Flip'):
        continue
    folder_path = os.path.join(DATASET_ROOT, folder)
    if os.path.isdir(folder_path):
        for file in os.listdir(folder_path):
            if file.endswith(".jpg"):
                all_image_paths.append(os.path.join(folder_path, file))

print(f"Total images found: {len(all_image_paths)}")

train_paths, temp_paths = train_test_split(all_image_paths, test_size=0.20, random_state=123)
val_paths, test_paths = train_test_split(temp_paths, test_size=0.50, random_state=123)

print(f"Split -> Train: {len(train_paths)} | Val: {len(val_paths)} | Test: {len(test_paths)}")

train_dataset = LP300W(image_paths=train_paths, transform=transforms_pipeline, occlusion_mode="random")
val_dataset = LP300W(image_paths=val_paths, transform=transforms_pipeline, occlusion_mode="random")

test_raw = LP300W(image_paths=test_paths, transform=transforms_pipeline, apply_occlusion=False)
test_mask = LP300W(image_paths=test_paths, transform=transforms_pipeline, occlusion_mode="mask")
test_glasses = LP300W(image_paths=test_paths, transform=transforms_pipeline, occlusion_mode="glasses")
test_both = LP300W(image_paths=test_paths, transform=transforms_pipeline, occlusion_mode="both")

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset,batch_size=BATCH_SIZE, shuffle=False)

raw_loader = DataLoader(test_raw, batch_size=BATCH_SIZE, shuffle=False)
mask_loader = DataLoader(test_mask, batch_size=BATCH_SIZE, shuffle=False)
glasses_loader = DataLoader(test_glasses, batch_size=BATCH_SIZE, shuffle=False)
both_loader = DataLoader(test_both, batch_size=BATCH_SIZE, shuffle=False)

Total images found: 61225
Split -> Train: 48980 | Val: 6122 | Test: 6123


In [4]:
sys.path.append("./models")

from token_hpe import TokenHPE

model = TokenHPE(depth=3)
checkpoint = torch.load("./weights/TokenHPEv1-ViTB-224_224-lyr3.tar", map_location="cpu")

state_dict = checkpoint["model_state_dict"]

model.load_state_dict(state_dict, strict=True)

model = model.to(device)
model.eval()

/home/anton/miniconda3/envs/cv2-2/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


==> Add Sine PositionEmbedding~


/tmp/ipykernel_104895/2510954716.py:6: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load("./weights/TokenHPEv1-ViTB-224_224-lyr3.tar", map_location="cpu"

TokenHPE(
  (feature_extractor): VisionTransformer(
    (patch_embed): PatchEmbed(
      (proj): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
      (norm): Identity()
    )
    (pos_drop): Dropout(p=0.0, inplace=False)
    (blocks): Sequential(
      (0): Block(
        (norm1): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
        (attn): Attention(
          (qkv): Linear(in_features=768, out_features=2304, bias=True)
          (attn_drop): Dropout(p=0.0, inplace=False)
          (proj): Linear(in_features=768, out_features=768, bias=True)
          (proj_drop): Dropout(p=0.0, inplace=False)
        )
        (drop_path): Identity()
        (norm2): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
        (mlp): Mlp(
          (fc1): Linear(in_features=768, out_features=3072, bias=True)
          (act): GELU(approximate='none')
          (fc2): Linear(in_features=3072, out_features=768, bias=True)
          (drop): Dropout(p=0.0, inplace=False)
        )
      )

In [5]:
print("\n-> Test 1/4: RAW")
yaw_r, pitch_r, roll_r, total_r = utils.evaluate_mae(model, raw_loader, device)

print("\n-> Test 2/4: MASK")
yaw_m, pitch_m, roll_m, total_m = utils.evaluate_mae(model, mask_loader, device)

print("\n-> Test 3/4: GLASSES")
yaw_g, pitch_g, roll_g, total_g = utils.evaluate_mae(model, glasses_loader, device)

print("\n-> Test 4/4: BOTH")
yaw_b, pitch_b, roll_b, total_b = utils.evaluate_mae(model, both_loader, device)

print("Test results:")
print(f"Test RAW     -> Yaw: {yaw_r} | Pitch: {pitch_r} | Roll: {roll_r} | Total: {total_r}")
print(f"Test MASK    -> Yaw: {yaw_m} | Pitch: {pitch_m} | Roll: {roll_m} | Total: {total_m}")
print(f"Test GLASSES -> Yaw: {yaw_g} | Pitch: {pitch_g} | Roll: {roll_g} | Total: {total_g}")
print(f"Test BOTH    -> Yaw: {yaw_b} | Pitch: {pitch_b} | Roll: {roll_b} | Total: {total_b}")


\n-> Test 1/4: RAW


Evaluando: 100%|██████████| 192/192 [00:53<00:00,  3.57it/s]


\n-> Test 2/4: MASK


Evaluando: 100%|██████████| 192/192 [00:54<00:00,  3.51it/s]


\n-> Test 3/4: GLASSES


Evaluando: 100%|██████████| 192/192 [00:54<00:00,  3.51it/s]


\n-> Test 4/4: BOTH


Evaluando: 100%|██████████| 192/192 [00:54<00:00,  3.51it/s]

Test results:
Test RAW     -> Yaw: 0.10456918930279241 | Pitch: 0.1476972696926398 | Roll: 0.16066439753748438 | Total: 0.13764361884430554
Test MASK    -> Yaw: 0.2901994369786568 | Pitch: 0.18760626444177225 | Roll: 0.20315286246036054 | Total: 0.22698618796026318
Test GLASSES -> Yaw: 0.14994004908562797 | Pitch: 0.18579966710825946 | Roll: 0.17383625346539983 | Total: 0.16985865655309573
Test BOTH    -> Yaw: 0.3141216805377233 | Pitch: 0.22062689014340117 | Roll: 0.21439691242508963 | Total: 0.2497151610354047


# Fine-Tunning

In [14]:
criterion = nn.L1Loss()

optimizer = optim.AdamW(model.parameters(), lr=5e-5, weight_decay=1e-4)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=3, factor=0.5, verbose=True)

EPOCHS = 5
best_val_loss = float('inf')

In [15]:
import time
from models.utils import compute_euler_angles_from_rotation_matrices

train_losses = []
val_losses = []

for epoch in range(EPOCHS):
    start_time = time.time()
    
    # Training
    model.train()
    running_train_loss = 0.0
    
    for imgs, poses in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]"):
        imgs, poses = imgs.to(device), poses.to(device)
        
        optimizer.zero_grad()
        
        predictions = model(imgs)
        
        if isinstance(predictions, (tuple, list)):
            pred_mat = predictions[0]
            pred_angles = compute_euler_angles_from_rotation_matrices(pred_mat, use_gpu=True)
            pred_angles = pred_angles.to(device)
        else:
            pred_angles = predictions
            
        loss = criterion(pred_angles, poses)
        
        loss.backward()
        optimizer.step()
        
        running_train_loss += loss.item() * imgs.size(0)
        
    epoch_train_loss = running_train_loss / len(train_dataset)
    train_losses.append(epoch_train_loss)
    
    # Validation
    model.eval()
    running_val_loss = 0.0
    
    with torch.no_grad():
        for imgs, poses in tqdm(val_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Val]"):
            imgs, poses = imgs.to(device), poses.to(device)
            
            predictions = model(imgs)
            
            if isinstance(predictions, (tuple, list)):
                pred_mat = predictions[0]
                pred_angles = compute_euler_angles_from_rotation_matrices(pred_mat, use_gpu=True)
                pred_angles = pred_angles.to(device)
            else:
                pred_angles = predictions
                
            loss = criterion(pred_angles, poses)
            running_val_loss += loss.item() * imgs.size(0)
            
    epoch_val_loss = running_val_loss / len(val_dataset)
    val_losses.append(epoch_val_loss)
    
    scheduler.step(epoch_val_loss)
    
    epoch_time = time.time() - start_time
    
    print(f"\\nEpoch {epoch+1}/{EPOCHS} completed in {epoch_time:.0f}s")
    print(f"Train Loss (MAE radians): {epoch_train_loss:.4f} | Val Loss: {epoch_val_loss:.4f}")
    
    if epoch_val_loss < best_val_loss:
        best_val_loss = epoch_val_loss
        save_path = "./weights/TokenHPE_Finetuned_Best.pth"
        torch.save(model.state_dict(), save_path)
        print("Best model updated")
    else:
        print("\\n")

Epoch 1/5 [Val]: 100%|██████████| 192/192 [00:54<00:00,  3.52it/s]


\nEpoch 1/5 completed in 1173s
Train Loss (MAE radians): 0.0524 | Val Loss: 0.0438
Best model updated


Epoch 2/5 [Val]: 100%|██████████| 192/192 [00:52<00:00,  3.68it/s]


\nEpoch 2/5 completed in 1164s
Train Loss (MAE radians): 0.0373 | Val Loss: 0.0339
Best model updated


Epoch 3/5 [Val]: 100%|██████████| 192/192 [00:51<00:00,  3.70it/s]


\nEpoch 3/5 completed in 1137s
Train Loss (MAE radians): 0.0320 | Val Loss: 0.0312
Best model updated


Epoch 4/5 [Val]: 100%|██████████| 192/192 [00:52<00:00,  3.69it/s]


\nEpoch 4/5 completed in 1137s
Train Loss (MAE radians): 0.0286 | Val Loss: 0.0300
Best model updated


Epoch 5/5 [Val]: 100%|██████████| 192/192 [00:51<00:00,  3.70it/s]


\nEpoch 5/5 completed in 1137s
Train Loss (MAE radians): 0.0270 | Val Loss: 0.0262
Best model updated


In [16]:
sys.path.append("./models")

from token_hpe import TokenHPE

model = TokenHPE(depth=3)
checkpoint_path = "./weights/TokenHPE_Finetuned_Best.pth"
checkpoint = torch.load(checkpoint_path, map_location="cpu")

state_dict = checkpoint.get("model_state_dict", checkpoint)
model.load_state_dict(state_dict, strict=True)

model = model.to(device)
model.eval()

print("\n-> Test 1/4: RAW")
yaw_r, pitch_r, roll_r, total_r = utils.evaluate_mae(model, raw_loader, device)

print("\n-> Test 2/4: MASK")
yaw_m, pitch_m, roll_m, total_m = utils.evaluate_mae(model, mask_loader, device)

print("\n-> Test 3/4: GLASSES")
yaw_g, pitch_g, roll_g, total_g = utils.evaluate_mae(model, glasses_loader, device)

print("\n-> Test 4/4: BOTH")
yaw_b, pitch_b, roll_b, total_b = utils.evaluate_mae(model, both_loader, device)

print("Test results:")
print(f"Test RAW     -> Yaw: {yaw_r} | Pitch: {pitch_r} | Roll: {roll_r} | Total: {total_r}")
print(f"Test MASK    -> Yaw: {yaw_m} | Pitch: {pitch_m} | Roll: {roll_m} | Total: {total_m}")
print(f"Test GLASSES -> Yaw: {yaw_g} | Pitch: {pitch_g} | Roll: {roll_g} | Total: {total_g}")
print(f"Test BOTH    -> Yaw: {yaw_b} | Pitch: {pitch_b} | Roll: {roll_b} | Total: {total_b}")


/tmp/ipykernel_104895/1973561996.py:7: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(checkpoint_path, map_location="cpu")


==> Add Sine PositionEmbedding~

-> Test 1/4: RAW


Evaluando: 100%|██████████| 192/192 [00:54<00:00,  3.55it/s]



-> Test 2/4: MASK


Evaluando: 100%|██████████| 192/192 [00:54<00:00,  3.52it/s]



-> Test 3/4: GLASSES


Evaluando: 100%|██████████| 192/192 [00:54<00:00,  3.52it/s]



-> Test 4/4: BOTH


Evaluando: 100%|██████████| 192/192 [00:54<00:00,  3.52it/s]

Test results:
Test RAW     -> Yaw: 0.028649591735700912 | Pitch: 0.03091817129559714 | Roll: 0.022918503404773374 | Total: 0.027495422145357145
Test MASK    -> Yaw: 0.02951787054665276 | Pitch: 0.02950996041745702 | Roll: 0.023453331516397637 | Total: 0.027493720826835807
Test GLASSES -> Yaw: 0.024817457982046473 | Pitch: 0.030718760379054237 | Roll: 0.02161380061619659 | Total: 0.025716672992432434
Test BOTH    -> Yaw: 0.02511102219101418 | Pitch: 0.027757176392299958 | Roll: 0.02239493296346146 | Total: 0.025087710515591864


# TODO list

## Cargar los 3 modelos

## Crear division de entrenamiento/test

## Finetunnear los 3 modelos

## Hacer test normal con los baselines y con los finetunned

## Hacer test con imagenes reales